In [16]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch
from pathlib import Path
from peft import LoraConfig, get_peft_model
import yaml
ROOT_DIR = Path().resolve().parent.parent


In [17]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.bfloat16)
model_4bit = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-1.7B-Base",quantization_config=quantization_config, dtype="auto", device_map="auto")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]c:\Users\Fabien\Desktop\OC\P14\FINE-TUNING_MEDICAL\.venv\Lib\site-packages\bitsandbytes\backends\default\ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
c:\Users\Fabien\Desktop\OC\P14\FINE-TUNING_MEDICAL\.venv\Lib\site-packages\bitsandbytes\backends\cpu\ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 310/310 [02:37<00:00,  1.96it/s]


In [18]:
with open(ROOT_DIR / "params.yaml", "r") as f:
    config = yaml.safe_load(f)['lora_config']

lora_config = LoraConfig(
    task_type=config['task_type'],
    r=config['r'],
    lora_alpha=config['lora_alpha'],
    lora_dropout=config['lora_dropout'],
    target_modules=config['target_modules']
)

model = get_peft_model(model_4bit, lora_config)
model.print_trainable_parameters()

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [19]:
import pandas as pd
df = pd.read_parquet(ROOT_DIR / "data/processed/sft_dataset/sft_train.parquet")
df.head()

,question,answer,medical_subject,has_clinical_case,language,question_type,confidence_level,dataset_name,token_count_question,token_count_answer
0,what is (are) prune belly syndrome ?,"prune belly syndrome, also called eagle-barret...",NaN,None,en,open_qa,high,medquad,9,191
1,what is (are) tietze syndrome ?,tietze syndrome is an inflammatory condition c...,NaN,None,en,open_qa,high,medquad,10,147
2,a <DATE> man from chile comes to the physician...,the patient's presentation includes a history ...,NaN,None,en,conversational,low,ultramed,195,383
3,"what are the symptoms of <LOCATION>, grebe type ?","what are the signs and symptoms of <LOCATION>,...",NaN,None,en,open_qa,high,medquad,12,545
4,in a patient with sle and concurrent mild live...,\n\nthe patient has sle (systemic lupus erythe...,NaN,None,en,conversational,low,ultramed,46,210
